# Week 1 — Ground Truth (Option C, Builder)

**Notes before reading**
- The official labels for items 1–5 were printed by `df.head()` before I labelled. I labelled them using my rules, but they are not fully blind.
- My category rules are in `README.md` (committed before any code).
- Code for the API calls, evaluation and Gradio interface was written with AI assistance. I ran and checked it.
- C7 (re-labelling my own data) is not done because of time.
- Results: `my_labels.csv`, `ai_prompt1.csv`, `ai_prompt2.csv`. The write-up is submitted separately.


In [1]:
import pandas as pd

df = pd.read_csv("week1_data.csv")

print(df.head())

    set  id                                               text  \
0  main   1  Gerrard only a 50-50 chance VIENNA - England m...   
1  main   2  A Better Solution for Israel The hysterical to...   
2  main   3  Olympic Daily Preview - Thursday, August 19 At...   
3  main   4  US men #39;s basketball routs Angola Athens, G...   
4  main   5  Italy: Sicilian derby ends in draw PALERMO, No...   

   official_number official_label  
0                2         Sports  
1                1          World  
2                2         Sports  
3                2         Sports  
4                2         Sports  


In [2]:
main, imbalanced, sorted_samples, full = df[df["set"] == "main"], df[df["set"] == "imbalanced"], df[df["set"] == "sorted_sample"], df[df["set"] == "full"]

print(main.head())
print(imbalanced.head())
print(sorted_samples.head())
print(full.head())

    set  id                                               text  \
0  main   1  Gerrard only a 50-50 chance VIENNA - England m...   
1  main   2  A Better Solution for Israel The hysterical to...   
2  main   3  Olympic Daily Preview - Thursday, August 19 At...   
3  main   4  US men #39;s basketball routs Angola Athens, G...   
4  main   5  Italy: Sicilian derby ends in draw PALERMO, No...   

   official_number official_label  
0                2         Sports  
1                1          World  
2                2         Sports  
3                2         Sports  
4                2         Sports  
           set  id                                               text  \
30  imbalanced   1  Tropical Storm Slams Into Coastal S.C. CHARLES...   
31  imbalanced   2  World ; Bigley: I want to live a simple life, ...   
32  imbalanced   3  In Taipei, Talk of Arms -- and Amity Premier Y...   
33  imbalanced   4  PM and Latham target Sydney Prime Minister Joh...   
34  imbalanced   5  Mo

In [3]:
print(len(main), len(imbalanced), len(sorted_samples), len(full))

30 30 200 7600


In [4]:
for i, row in main.iterrows():
  print(row["id"], row["text"])

1 Gerrard only a 50-50 chance VIENNA - England midfielder Steven Gerrard is a major doubt to face Austria in today #39;s opening 2006 World Cup qualifier. Gerrard has failed to shake off a groin injury suffered in training on Thursday and
2 A Better Solution for Israel The hysterical tone of Daniel Seidemann #39;s plea to the next US administration to save Israel from itself serves no useful purpose op-ed, Aug. 26.
3 Olympic Daily Preview - Thursday, August 19 Athens, Greece (Sports Network) - Wednesday night it was Paul Hamm #39;s turn to shine for the United States, as he won the gold medal in the men #39;s all-around competition. Will Thursday produce a sweep for the US at the Olympics? ...
4 US men #39;s basketball routs Angola Athens, Greece (Sports Network) - Tim Duncan led a balanced American attack with 15 points and seven rebounds, as the United States men #39;s basketball team completed the preliminary round with a resounding 89-53 victory over winless Angola.
5 Italy: Sicili

## Hand labels (30 items, written before seeing official labels for items 6–30)

In [5]:
my_labels = [
    "Sports",    # 1
    "World",     # 2
    "Sports",       # 3
    "Sports",     # 4
    "Sports",     # 5
    "Sci/Tech",     # 6
    "World",     # 7
    "Business",     # 8
    "World",     # 9
    "Business",     # 10
    "World",     # 11
    "World",     # 12
    "Sci/Tech",     # 13
    "Sci/Tech",     # 14
    "Business",     # 15
    "Business",     # 16
    "World",     # 17
    "Sci/Tech",     # 18
    "Sci/Tech",     # 19
    "World",     # 20
    "World",     # 21
    "Business",     # 22
    "Business",     # 23
    "World",     # 24
    "World",     # 25
    "World",     # 26
    "Sports",     # 27
    "Sports",     # 28
    "Business",     # 29
    "Sci/Tech",     # 30

]

my_confidence = [
    "H",   # 1
    "H",   # 2
    "H", # 3
    "L",   # 4
    "H",   # 5
    "M",   # 6
    "H",   # 7
    "H",   # 8
    "M",   # 9
    "H",   #10
    "H",   #11
    "H",   #12
    "M",   #13
    "H",   #14
    "H",   #15
    "H",   #16
    "H",   #17
    "L",   #18
    "M",   #19
    "H",   #20
    "L",   #21
    "M",   #22
    "M",   #23
    "M",   #24
    "M",   #25
    "L",   #26
    "H",   #27
    "M",   #28
    "M",   #29
    "M",   #30
]

In [6]:
for i, row in main.iterrows():
    print(row["id"], my_labels[i], my_confidence[i], "|", row["text"][:80])

1 Sports H | Gerrard only a 50-50 chance VIENNA - England midfielder Steven Gerrard is a majo
2 World H | A Better Solution for Israel The hysterical tone of Daniel Seidemann #39;s plea 
3 Sports H | Olympic Daily Preview - Thursday, August 19 Athens, Greece (Sports Network) - We
4 Sports L | US men #39;s basketball routs Angola Athens, Greece (Sports Network) - Tim Dunca
5 Sports H | Italy: Sicilian derby ends in draw PALERMO, Nov 11 (SW) - After a 39-year wait f
6 Sci/Tech M | 5 of arthritis patients in Singapore take Bextra or Celebrex &lt;b&gt;...&lt;/b&
7 World H | Prince Charles chastised for quot;old fashioned quot; views A minister has launc
8 Business H | Dollar Idles vs. Euro Before U.S. Data LONDON (Reuters) - The dollar kept close 
9 World M | Infineon to Pay a Fine in the Fixing Of Chip Prices Federal prosecutors announce
10 Business H | Images: Nintendo grows up--a little The Nintendo DS includes a touch-sensitive s
11 World H | Berlin ties Miami mark RALEIGH, N.C. -- Bro

In [7]:
assert len(my_labels) == len(my_confidence) == len(main)
assert all(x in ["Sports", "Sci/Tech", "Business", "World"] for x in my_labels)

In [8]:
import requests
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")
URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "openai/gpt-oss-120b"

text = main["text"][0]
prompt = "Classify this news item as World, Sports, Business or Sci/Tech:\n\n" + text

r = requests.post(
    URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": MODEL,
        "temperature": 0,        # 0 = the most predictable answer, no randomness
        "max_tokens": 300,       # upper limit on how long the reply can be
        "messages": [{"role": "user", "content": prompt}],
    },
    timeout=60,
)

print(r.status_code)
print(r.json())

200
{'id': 'chatcmpl-525fb059-8037-4ac4-b43c-920116bf6ea7', 'object': 'chat.completion', 'created': 1792074672, 'model': 'openai/gpt-oss-120b', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Sports', 'reasoning': "We need to classify the news item into categories: World, Sports, Business, Sci/Tech. The content is about Steven Gerrard, a football (soccer) player, doubt for a match. That's Sports."}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'queue_time': 0.11168602, 'prompt_tokens': 143, 'prompt_time': 0.007126307, 'completion_tokens': 55, 'completion_time': 0.116722654, 'total_tokens': 198, 'total_time': 0.123848961, 'completion_tokens_details': {'reasoning_tokens': 45}}, 'usage_breakdown': None, 'system_fingerprint': 'fp_4de1d6a1ee', 'x_groq': {'id': 'req_01m4zzjsn3ebr8wq2h2x5gm7hj', 'seed': 178429913}, 'service_tier': 'on_demand'}


## C2 — Structured JSON output that fails loudly

In [9]:
import json, time, requests
import pandas as pd
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")
URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "openai/gpt-oss-120b"
CATS = ["World", "Sports", "Business", "Sci/Tech"]

In [10]:
PROMPT_1 = """You are classifying news items into exactly one of four categories.

World: international news, politics, conflict, disasters, national affairs
Sports: matches, results, athletes, teams, transfers, tournaments
Business: companies, markets, earnings, the economy, trade, jobs, deals
Sci/Tech: science, research, technology, software, space, health research

Boundary rules, apply them in this order:
1. A foreign company's earnings or a currency move is Business, not World.
2. A broadcasting rights deal or a club's finances is Business, not Sports.
3. A product's technical capability is Sci/Tech. Its commercial impact is Business.
4. If two categories still fit, choose the one the item is mostly about.

Reply with JSON only, no other text, in exactly this form:
{{"label": "<World|Sports|Business|Sci/Tech>", "confidence": <number between 0 and 1>}}

News item:
{text}"""

PROMPT_2 = """What category is this news item? Pick one: World, Sports, Business, Sci/Tech.
Reply as JSON with keys "label" and "confidence".

News item:
{text}"""

In [11]:
class RateLimited(Exception):
    def __init__(self, wait):
        self.wait = wait

def call_model(prompt):
    r = requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": MODEL, "temperature": 0, "max_tokens": 300,
              "messages": [{"role": "user", "content": prompt}]},
        timeout=60,
    )
    if r.status_code == 429:                                   # free-tier limit hit
        raise RateLimited(float(r.headers.get("retry-after", 5)))
    r.raise_for_status()                                       # any other HTTP error -> exception
    return r.json()["choices"][0]["message"]["content"] or ""

def parse_reply(reply):
    """Return (label, confidence, error). Bad output is recorded as a failure, never repaired."""
    try:
        data = json.loads(reply)
    except (json.JSONDecodeError, TypeError):
        return None, None, "not valid JSON"
    if not isinstance(data, dict):
        return None, None, "JSON is not an object"
    label, conf = data.get("label"), data.get("confidence")
    if label not in CATS:
        return None, None, f"invalid label: {label!r}"
    if not isinstance(conf, (int, float)) or not 0 <= conf <= 1:
        return None, None, f"invalid confidence: {conf!r}"
    return label, float(conf), None

def classify_all(texts, template, pause=4.0, retries=4):
    rows = []
    for i, text in enumerate(texts):
        reply, err = None, None
        for _ in range(retries):
            try:
                reply = call_model(template.format(text=text))
                break
            except RateLimited as e:
                time.sleep(max(e.wait, pause))
            except requests.RequestException as e:
                err = f"request failed: {e}"
                break
        if reply is None:
            label, conf, err = None, None, err or "rate limited, gave up"
        else:
            label, conf, err = parse_reply(reply)
        rows.append({"label": label, "confidence": conf, "error": err, "raw": reply})
        if i < len(texts) - 1:
            time.sleep(pause)                                  # stay under 30 req / 8,000 tokens a minute
    return pd.DataFrame(rows)

# quick check on one item before spending 30 calls
print(parse_reply(call_model(PROMPT_1.format(text=main["text"][0]))))

('Sports', 0.99, None)


In [12]:
ai1 = classify_all(main["text"].tolist(), PROMPT_1)
ai2 = classify_all(main["text"].tolist(), PROMPT_2)

for name, res in [("prompt 1", ai1), ("prompt 2", ai2)]:
    print(f"{name}: parse failure rate = {res['label'].isna().mean():.0%}")
    print(res[res["label"].isna()][["error", "raw"]], "\n")

pd.DataFrame({"id": main["id"], "my_label": my_labels, "my_confidence": my_confidence}).to_csv("my_labels.csv", index=False)
ai1.assign(id=main["id"]).to_csv("ai_prompt1.csv", index=False)
ai2.assign(id=main["id"]).to_csv("ai_prompt2.csv", index=False)

prompt 1: parse failure rate = 3%
             error raw
15  not valid JSON     

prompt 2: parse failure rate = 0%
Empty DataFrame
Columns: [error, raw]
Index: [] 



## C3 — Agreement, kappa, confusion matrix

In [13]:
from sklearn.metrics import cohen_kappa_score, classification_report

gold = main["official_label"].tolist()
mine = my_labels
a1 = ai1["label"].fillna("FAILED").tolist()
a2 = ai2["label"].fillna("FAILED").tolist()

def agree(a, b):
    return sum(x == y for x, y in zip(a, b)) / len(a)

pairs = {
    "me vs AI (prompt 1)":       (mine, a1),
    "me vs official":            (mine, gold),
    "AI (prompt 1) vs official": (a1, gold),
    "me vs AI (prompt 2)":       (mine, a2),
    "AI (prompt 2) vs official": (a2, gold),
}
for name, (a, b) in pairs.items():
    print(f"{name:28} agreement {agree(a, b):.2f}   kappa {cohen_kappa_score(a, b):.2f}")

me vs AI (prompt 1)          agreement 0.67   kappa 0.56
me vs official               agreement 0.60   kappa 0.47
AI (prompt 1) vs official    agreement 0.83   kappa 0.78
me vs AI (prompt 2)          agreement 0.70   kappa 0.60
AI (prompt 2) vs official    agreement 0.83   kappa 0.78


In [14]:
print(pd.crosstab(pd.Series(gold, name="official"), pd.Series(a1, name="AI prompt 1")), "\n")
print(pd.crosstab(pd.Series(gold, name="official"), pd.Series(mine, name="me")), "\n")
print(classification_report(gold, a1, labels=CATS, zero_division=0))

cmp = main[["id", "text"]].assign(me=mine, my_conf=my_confidence, ai=a1, official=gold)

# Written question 3: you and the AI agree, the official label disagrees with both
print(cmp[(cmp.me == cmp.ai) & (cmp.me != cmp.official)][["id", "me", "official", "text"]].to_string())

# Calibration: were your Low-confidence items the ones you got wrong?
print(pd.crosstab(cmp.my_conf, cmp.me == cmp.official, margins=True))

AI prompt 1  Business  FAILED  Sci/Tech  Sports  World
official                                              
Business            6       0         1       0      0
Sci/Tech            2       0         4       0      1
Sports              0       0         0       8      0
World               0       1         0       0      7 

me        Business  Sci/Tech  Sports  World
official                                   
Business         3         2       1      1
Sci/Tech         3         3       0      1
Sports           0         1       5      2
World            1         0       0      7 

              precision    recall  f1-score   support

       World       0.88      0.88      0.88         8
      Sports       1.00      1.00      1.00         8
    Business       0.75      0.86      0.80         7
    Sci/Tech       0.80      0.57      0.67         7

   micro avg       0.86      0.83      0.85        30
   macro avg       0.86      0.83      0.84        30
weighted avg       0.8

## C3 — TF-IDF baseline, leakage, imbalanced set

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def tfidf():
    return make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))

# Baseline trained on the full 7,600 rows, scored on a held-out 25%
Xtr, Xte, ytr, yte = train_test_split(full["text"], full["official_label"], test_size=0.25,
                                      random_state=0, stratify=full["official_label"])
base = tfidf().fit(Xtr, ytr)
print("TF-IDF held-out accuracy (full):", round(accuracy_score(yte, base.predict(Xte)), 3))
print("TF-IDF on the 30 main items:   ", round(agree(list(base.predict(main["text"])), gold), 3))

# Leakage on the small 200-row pool
Ptr, Pte, qtr, qte = train_test_split(sorted_samples["text"], sorted_samples["official_label"], test_size=0.3,
                                      random_state=0, stratify=sorted_samples["official_label"])
honest = tfidf().fit(Ptr, qtr)
leaky = tfidf().fit(sorted_samples["text"], sorted_samples["official_label"])   # test rows included in training
print("honest, held out:       ", round(accuracy_score(qte, honest.predict(Pte)), 3))
print("leaky, seen the answers:", round(accuracy_score(qte, leaky.predict(Pte)), 3))

# Imbalanced set: answer World every time
truth = imbalanced["official_label"].tolist()
always_world = ["World"] * len(truth)
print("imbalanced class counts:", imbalanced["official_label"].value_counts().to_dict())
print("always-World accuracy:", agree(always_world, truth), " kappa:", cohen_kappa_score(always_world, truth))

TF-IDF held-out accuracy (full): 0.865
TF-IDF on the 30 main items:    0.933
honest, held out:        0.617
leaky, seen the answers: 1.0
imbalanced class counts: {'World': 24, 'Sci/Tech': 2, 'Business': 2, 'Sports': 2}
always-World accuracy: 0.8  kappa: 0.0


## C4 — Confidence interval

In [16]:
from statsmodels.stats.proportion import proportion_confint

k = sum(x == y for x, y in zip(a1, gold))
lo, hi = proportion_confint(k, len(gold), method="wilson")
print(f"AI vs official: {k}/{len(gold)} = {k/len(gold):.0%}, 95% CI {lo:.0%} to {hi:.0%}")

p = k / len(gold)
n_needed = 1.96**2 * p * (1 - p) / 0.05**2
print(f"Items needed for a ±5-point interval: about {int(n_needed) + 1}")

AI vs official: 25/30 = 83%, 95% CI 66% to 93%
Items needed for a ±5-point interval: about 214


## C5 — Trying to break it

In [17]:
tests = {
    "empty string": "",
    "3,000 words of noise": " ".join(["banana quantum lorem"] * 1000),
    "another language": "Le Real Madrid a battu Barcelone 3-1 samedi soir au Bernabéu.",
    "two categories": "Manchester United shares fall 8% after the club misses Champions League qualification.",
}
for name, t in tests.items():
    reply = call_model(PROMPT_1.format(text=t))
    print(f"{name:22} -> {parse_reply(reply)}   raw: {reply[:80]!r}")
    time.sleep(4)

# Fix: an empty input should fail loudly before any API call, not get a made-up label
def classify_safe(text):
    if not text or not text.strip():
        return None, None, "empty input rejected"
    return parse_reply(call_model(PROMPT_1.format(text=text)))

print("after fix:", classify_safe(""))

empty string           -> ('World', 0.0, None)   raw: '{"label": "World", "confidence": 0.0}'
3,000 words of noise   -> ('Sci/Tech', 0.9, None)   raw: '{"label": "Sci/Tech", "confidence": 0.9}'
another language       -> ('Sports', 0.99, None)   raw: '{"label": "Sports", "confidence": 0.99}'
two categories         -> ('Business', 0.97, None)   raw: '{"label": "Business", "confidence": 0.97}'
after fix: (None, None, 'empty input rejected')


## C6 — Gradio interface

In [18]:
!pip -q install gradio
import gradio as gr

def predict(text):
    label, conf, err = classify_safe(text)
    return f"FAILED: {err}" if err else f"{label} (confidence {conf:.2f})"

gr.Interface(fn=predict,
             inputs=gr.Textbox(lines=4, label="News item"),
             outputs=gr.Textbox(label="Category"),
             title="AG News classifier").launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7cff2bfdf667eb6b33.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Pinned requirements

In [19]:
import importlib.metadata as md
pkgs = ["pandas", "requests", "scikit-learn", "statsmodels", "gradio"]
open("requirements.txt", "w").write("\n".join(f"{p}=={md.version(p)}" for p in pkgs))
print(open("requirements.txt").read())

pandas==2.2.3
requests==2.32.4
scikit-learn==1.6.1
statsmodels==0.15.0
gradio==6.29.0


## Stack questions — retrieval and SQL

In [ ]:
# Retrieval: keyword search for "world" in the 30 items
hits = main[main["text"].str.contains("world", case=False)]
print(hits[["id", "official_label", "text"]].to_string())

In [ ]:
# The same data as SQL (DuckDB runs SQL directly on a pandas DataFrame)
import duckdb
duckdb.query("""
    SELECT official_label, COUNT(*) AS n, ROUND(AVG(LENGTH(text)), 1) AS avg_chars
    FROM main GROUP BY official_label ORDER BY n DESC
""").df()